# W9-D6 概念实验：对象关系、生命周期与依赖

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：对象之间应按“引用”还是“持有内容”连接？**

用一个有向依赖表表示 Week 9 核心链，检查是否存在违反制品链方向的反向边。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
layers = {"DigitalEmployeeDefinition": 0, "BlueprintVersion": 1, "SkillRelease": 1, "ReleaseChannel": 1, "Deployment": 2, "DeploymentRevision": 2, "TrafficPolicy": 2, "Execution": 2}
edges = [("DigitalEmployeeDefinition", "BlueprintVersion"), ("BlueprintVersion", "SkillRelease"), ("SkillRelease", "DeploymentRevision"), ("Deployment", "DeploymentRevision"), ("DeploymentRevision", "TrafficPolicy"), ("TrafficPolicy", "Execution")]
for left, right in edges: print(f"{left:25} -> {right}")
backward = [(a, b) for a, b in edges if layers[a] > layers[b]]
print("跨层反向依赖：", backward)
assert not backward
print("说明：上游制品可被下游精确引用；运行时不反向编辑源制品。")

## 实验问题

**问题 2：哪些对象的生命周期节奏不同，因而不应合并？**

将常见变更频率量化：员工定义按季度、release 按周、traffic 按小时。频率差越大，合并造成的无关变更越多。

In [ ]:
objects = ["员工定义", "BlueprintVersion", "SkillRelease", "DeploymentRevision", "TrafficPolicy"]
changes_per_month = np.array([0.2, 1, 4, 8, 30])
fig, ax = plt.subplots(figsize=(7.4, 3.6))
ax.bar(objects, changes_per_month, color=["#7570b3", "#1b9e77", "#66a61e", "#e7298a", "#d95f02"])
ax.set_ylabel("典型变更次数/月"); ax.set_title("演化节奏不同，是对象分离的可观察信号")
ax.tick_params(axis="x", rotation=20); plt.tight_layout(); plt.show()
print(dict(zip(objects, changes_per_month)))
assert changes_per_month.max() / changes_per_month.min() > 100

## 实验问题

**问题 3：生命周期合法性可以机械检查吗？**

为候选、版本、release 与部署定义最小状态机，拒绝跳过评审或直接让草案进入运行时的转换。

In [ ]:
allowed = {
    "candidate": {"draft": {"in_review", "rejected"}, "in_review": {"promoted", "rejected"}},
    "release": {"built": {"evaluated"}, "evaluated": {"approved"}, "approved": {"published"}},
    "deployment": {"draft": {"active", "retired"}, "active": {"suspended", "retired"}},
}
def transition(kind, old, new):
    if new not in allowed[kind].get(old, set()): raise ValueError(f"非法转换：{kind} {old}->{new}")
    return new
print("合法：", transition("candidate", "in_review", "promoted"))
try: transition("release", "built", "published")
except ValueError as exc: print(exc)
try: transition("deployment", "draft", "suspended")
except ValueError as exc: print(exc)

## 实验问题

**问题 4：合并 Channel 和 TrafficPolicy 会损失什么状态？**

枚举四种合法组合：新版本可以已晋升但尚未切流，说明两个对象不是冗余副本。

In [ ]:
states = [("old", "old"), ("new", "old"), ("new", "canary"), ("new", "new")]
labels = ["未晋升", "已晋升\n未切流", "灰度", "全量"]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(range(4), [0, 1, 2, 3], s=180, color="#1b9e77")
for x, (channel_state, traffic_state) in enumerate(states):
    ax.text(x, x + .12, f"Channel={channel_state}\nTraffic={traffic_state}", ha="center", fontsize=9)
ax.set_xticks(range(4), labels); ax.set_yticks([]); ax.set_title("晋升指针与流量策略组合出独立、必要的状态")
plt.tight_layout(); plt.show()
assert states[1] == ("new", "old")
print("“已晋升但未切流”是灰度治理所必需的中间状态。")